# Provided code

In [ ]:
mode = "colab" # change this to "local" if you are on your own computer

if mode == "local":
    import models
elif mode == "colab":
    import requests
    url = 'https://github.com/ahmadianlab/gg3_nda/blob/main/models.py?raw=true'
    r = requests.get(url)
    with open('models.py', 'w') as f:
        f.write(r.text)
    import models
else:
    raise Exception("mode must be either local or colab")

In [ ]:
import numpy as np
import numpy.random as npr


def lo_histogram(x, bins):
    """
    Left-open version of np.histogram with left-open bins covering the interval (left_edge, right_edge]
    (np.histogram does the opposite and treats bins as right-open.)
    Input & output behaviour is exactly the same as np.histogram
    """
    out = np.histogram(-x, -bins[::-1])
    return out[0][::-1], out[1:]


def gamma_isi_point_process(rate, shape):
    """
    Simulates (1 trial of) a sub-poisson point process (with underdispersed inter-spike intervals relative to Poisson)
    :param rate: time-series giving the mean spike count (firing rate * dt) in different time bins (= time steps)
    :param shape: shape parameter of the gamma distribution of ISI's
    :return: vector of spike counts with same shape as "rate".
    """
    sum_r_t = np.hstack((0, np.cumsum(rate)))
    gs = np.zeros(2)
    while gs[-1] < sum_r_t[-1]:
        gs = np.cumsum( npr.gamma(shape, 1 / shape, size=(2 + int(2 * sum_r_t[-1]),)) )
    y, _ = lo_histogram(gs, sum_r_t)

    return y



class StepModel():
    """
    Simulator of the Stepping Model of Latimer et al. Science 2015.
    """
    def __init__(self, m=50, r=10, x0=0.2, Rh=50, isi_gamma_shape=None, Rl=None, dt=None):
        """
        Simulator of the Stepping Model of Latimer et al. Science 2015.
        :param m: mean jump time (in # of time-steps). This is the mean parameter of the Negative Binomial distribution
                  of jump (stepping) time
        :param r: parameter r ("# of successes") of the Negative Binomial (NB) distribution of jump (stepping) time
                  (Note that it is more customary to parametrise the NB distribution by its parameter p and r,
                  instead of m and r, where p is so-called "probability of success" (see Wikipedia). The two
                  parametrisations are equivalent and one can go back-and-forth via: m = r (1-p)/p and p = r / (m + r).)
        :param x0: determines the pre-jump firing rate, via  R_pre = x0 * Rh (see below for Rh)
        :param Rh: firing rate of the "up" state (the same as the post-jump state in most of the project tasks)
        :param isi_gamma_shape: shape parameter of the Gamma distribution of inter-spike intervals.
                            see https://en.wikipedia.org/wiki/Gamma_distribution
        :param Rl: firing rate of the post-jump "down" state (rarely used)
        :param dt: real time duration of time steps in seconds (only used for converting rates to units of inverse time-step)
        """
        self.m = m
        self.r = r
        self.x0 = x0

        self.p = r / (m + r)

        self.Rh = Rh
        if Rl is not None:
            self.Rl = Rl

        self.isi_gamma_shape = isi_gamma_shape
        self.dt = dt


    @property
    def params(self):
        return self.m, self.r, self.x0

    @property
    def fixed_params(self):
        return self.Rh, self.Rl


    def emit(self, rate):
        """
        emit spikes based on rates
        :param rate: firing rate sequence, r_t, possibly in many trials. Shape: (Ntrials, T)
        :return: spike train, n_t, as an array of shape (Ntrials, T) containing integer spike counts in different
                 trials and time bins.
        """
        if self.isi_gamma_shape is None:
            # poisson spike emissions
            y = npr.poisson(rate * self.dt)
        else:
            # sub-poisson/underdispersed spike emissions
            y = gamma_isi_point_process(rate * self.dt, self.isi_gamma_shape)

        return y


    def simulate(self, Ntrials=1, T=100, get_rate=True):
        """
        :param Ntrials: (int) number of trials
        :param T: (int) duration of each trial in number of time-steps.
        :param get_rate: whether or not to return the rate time-series
        :return:
        spikes: shape = (Ntrial, T); spikes[j] gives the spike train, n_t, in trial j, as
                an array of spike counts in each time-bin (= time step)
        jumps:  shape = (Ntrials,) ; jumps[j] is the jump time (aka step time), tau, in trial j.
        rates:  shape = (Ntrial, T); rates[j] is the rate time-series, r_t, in trial j (returned only if get_rate=True)
        """
        # set dt (time-step duration in seconds) such that trial duration is always 1 second, regardless of T.
        dt = 1 / T
        self.dt = dt

        ts = np.arange(T)

        spikes, jumps, rates = [], [], []
        for tr in range(Ntrials):
            # sample jump time
            jump = npr.negative_binomial(self.r, self.p)
            jumps.append(jump)

            # first set rate at all times to pre-step rate
            rate = np.ones(T) * self.x0 * self.Rh
            # then set rates after jump to self.Rh
            rate[ts >= jump] = self.Rh
            rates.append(rate)

            spikes.append(self.emit(rate))

        if get_rate:
            return np.array(spikes), np.array(jumps), np.array(rates)
        else:
            return np.array(spikes), np.array(jumps)


class RampModel():
    """
    Simulator of the Ramping Model (aka Drift-Diffusion Model) of Latimer et al., Science (2015).
    """
    def __init__(self, beta=0.5, sigma=0.2, x0=.2, Rh=50, isi_gamma_shape=None, Rl=None, dt=None):
        """
        Simulator of the Ramping Model of Latimer et al. Science 2015.
        :param beta: drift rate of the drift-diffusion process
        :param sigma: diffusion strength of the drift-diffusion process.
        :param x0: average initial value of latent variable x[0]
        :param Rh: the maximal firing rate obtained when x_t reaches 1 (corresponding to the same as the post-step
                   state in most of the project tasks)
        :param isi_gamma_shape: shape parameter of the Gamma distribution of inter-spike intervals.
                            see https://en.wikipedia.org/wiki/Gamma_distribution
        :param Rl: Not implemented. Ignore.
        :param dt: real time duration of time steps in seconds (only used for converting rates to units of inverse time-step)
        """
        self.beta = beta
        self.sigma = sigma
        self.x0 = x0

        self.Rh = Rh
        if Rl is not None:
            self.Rl = Rl

        self.isi_gamma_shape = isi_gamma_shape
        self.dt = dt


    @property
    def params(self):
        return self.mu, self.sigma, self.x0

    @property
    def fixed_params(self):
        return self.Rh, self.Rl


    def f_io(self, xs, b=None):
        if b is None:
            return self.Rh * np.maximum(0, xs)
        else:
            return self.Rh * b * np.log(1 + np.exp(xs / b))


    def emit(self, rate):
        """
        emit spikes based on rates
        :param rate: firing rate sequence, r_t, possibly in many trials. Shape: (Ntrials, T)
        :return: spike train, n_t, as an array of shape (Ntrials, T) containing integer spike counts in different
                 trials and time bins.
        """
        if self.isi_gamma_shape is None:
            # poisson spike emissions
            y = npr.poisson(rate * self.dt)
        else:
            # sub-poisson/underdispersed spike emissions
            y = gamma_isi_point_process(rate * self.dt, self.isi_gamma_shape)

        return y


    def simulate(self, Ntrials=1, T=100, get_rate=True):
        """
        :param Ntrials: (int) number of trials
        :param T: (int) duration of each trial in number of time-steps.
        :param get_rate: whether or not to return the rate time-series
        :return:
        spikes: shape = (Ntrial, T); spikes[j] gives the spike train, n_t, in trial j, as
                an array of spike counts in each time-bin (= time step)
        xs:     shape = (Ntrial, T); xs[j] is the latent variable time-series x_t in trial j
        rates:  shape = (Ntrial, T); rates[j] is the rate time-series, r_t, in trial j (returned only if get_rate=True)
        """
        # set dt (time-step duration in seconds) such that trial duration is always 1 second, regardless of T.
        dt = 1 / T
        self.dt = dt

       # simulate all trials in parallel (using numpy arrays and broadcasting)

        # first, directly integrate/sum the drift-diffusion updates
        # x[t+1] = x[t] + β dt + σ √dt * randn (with initial condition x[0] = x0 + σ √dt * randn)
        # to get xs in shape (Ntrials, T):
        ts = np.arange(T)
        xs = self.x0 + self.beta * dt * ts + self.sigma * np.sqrt(dt) * np.cumsum(npr.randn(Ntrials, T), axis=1)
        # in each trial set x to 1 after 1st passage through 1; padding xs w 1 assures passage does happen, possibly at T+1
        taus = np.argmax(np.hstack((xs, np.ones((xs.shape[0],1)))) >= 1., axis=-1)
        xs = np.where(ts[None,:] >= taus[:,None], 1., xs)
        # # the above 2 lines are equivalent to:
        # for x in xs:
        #     if np.sum(x >= 1) > 0:
        #         tau = np.nonzero(x >= 1)[0][0]
        #         x[tau:] = 1

        rates = self.f_io(xs) # shape = (Ntrials, T)

        spikes = np.array([self.emit(rate) for rate in rates]) # shape = (Ntrial, T)

        if get_rate:
            return spikes, xs, rates
        else:
            return spikes, xs

# Task 1.4 - classifier using PSTH

The code below generates PSTH plots from step and ramp models for a variety of parameter choices. Also calculates the range of the second derivative (ie max - min values) within a specified timeframe, which is the metric used in the rule based classifier (although there are probably better metrics to use)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d


N_TRIALS = 5000
T_BINS   = 1000
SIGMA    = 50

t_lo, t_hi = 0.1, 0.90 # looking within a certain time range

def analyse(spike_trains, time, title):
    """Return smoothed psth, derivatives and the 2nd-derivative range."""
    dt = time[1] - time[0]
    psth = spike_trains.mean(axis=0)
    psth_smooth = gaussian_filter1d(psth, sigma=SIGMA)
    d1 = np.gradient(psth_smooth, dt)
    d2 = np.gradient(d1, dt)

    # range of 2nd derivative in the chosen window
    win = (time >= t_lo) & (time <= t_hi)
    d2_range = np.ptp(d2[win])
    print(f"{title:<35s}  Range = {d2_range:7.3f}")

    # plotting
    fig, axs = plt.subplots(1, 3, figsize=(15, 4))
    axs[0].plot(time, psth, label='raw')
    axs[0].plot(time, psth_smooth, '--', label='smoothed')
    axs[0].set_title(title); axs[0].legend()
    axs[0].set(xlabel='time (s)', ylabel='mean spike count')

    axs[1].plot(time, d1); axs[1].set_title('1st derivative')
    axs[1].set(xlabel='time (s)', ylabel='rate of change')

    axs[2].plot(time, d2); axs[2].set_title('2nd derivative')
    axs[2].set(xlabel='time (s)', ylabel='curvature')

    fig.tight_layout()
    plt.show()

    return d2_range



for m in [200, 500, 800]:
    for r in [3, 30, 300]:
        step_model = StepModel(m=m, r=r)
        spikes, *_ = step_model.simulate(Ntrials=N_TRIALS, T=T_BINS)

        t = np.linspace(0, 1, T_BINS)
        analyse(spikes, t, f"Step PSTH  m={m}, r={r}")


for beta in [0.3, 1, 3]:
    for sigma in [0.05, 0.15, 0.3]:
        ramp_model = RampModel(beta=beta, sigma=sigma)
        spikes, *_ = ramp_model.simulate(Ntrials=N_TRIALS, T=T_BINS)

        t = np.linspace(0, 1, T_BINS)
        analyse(spikes, t, f"Ramp PSTH β={beta}, σ={sigma}")


This is the simple rule based classifier. It uses the second derivative range as a metric (referred to as sharpness). It computes a threshold based on the training data. It is successful on 13/18 of the provided cases, (which I think is statistically significant with p=0.05), but it doesn't do well for ramp model with high beta (because it looks too much like a step), and also fails for the step model when r = 3 and the mean jump time is not super early, because the jump times are spread enough that it just looks like a ramp (see plots above).

I guess if we are going to use this we might have to use lots of training data with a variety of parameters to find an ideal sharpness threshold value that we can just apply.

In [ ]:
import numpy as np
from scipy.ndimage import gaussian_filter1d

T_BINS         = 1000
DT             = 1 / T_BINS
TIME           = np.linspace(0, 1, T_BINS)
T_LO, T_HI     = 0, 1
SMOOTH_SIGMA   = 50
CAL_NTRIALS    = 5000
TEST_NTRIALS   = 5000

def _sharpness(psth, sigma=SMOOTH_SIGMA):
    """Peak-to-peak (max−min) of the 2nd derivative in [T_LO, T_HI]."""
    psth_s = gaussian_filter1d(psth, sigma=sigma)
    d1     = np.gradient(psth_s, DT)
    d2     = np.gradient(d1,    DT)
    win    = (TIME >= T_LO) & (TIME <= T_HI)
    return np.ptp(d2[win])

def classify_spike_trains(spike_trains, threshold, sigma=SMOOTH_SIGMA):
    """Return ('step' | 'ramp'), sharpness_value."""
    psth = spike_trains.mean(axis=0)
    shp  = _sharpness(psth, sigma)
    return ('step' if shp > threshold else 'ramp'), shp


step_vals, ramp_vals = [], []

for m in [200, 500]:
    for r in [3, 30]:
        model = StepModel(m=m, r=r)
        spikes, *_ = model.simulate(Ntrials=CAL_NTRIALS, T=T_BINS)
        step_vals.append(_sharpness(spikes.mean(axis=0)))

for beta in [0.3, 1]:
    for sig in [0.05, 0.15]:
        model = RampModel(beta=beta, sigma=sig)
        spikes, *_ = model.simulate(Ntrials=CAL_NTRIALS, T=T_BINS)
        ramp_vals.append(_sharpness(spikes.mean(axis=0)))

threshold = (np.median(step_vals) + np.median(ramp_vals)) / 2
print(f"→ Automatic threshold   θ = {threshold:.3f}\n")


results = []

# Step
for m in [200, 500, 800]:
    for r in [3, 30, 300]:
        model = StepModel(m=m, r=r)
        spikes, *_ = model.simulate(Ntrials=TEST_NTRIALS, T=T_BINS)
        pred, shp = classify_spike_trains(spikes, threshold)
        results.append(("step", pred, shp, f"m={m}, r={r}"))

# Ramp
for beta in [0.3, 1, 3]:
    for sig in [0.05, 0.15, 0.3]:
        model = RampModel(beta=beta, sigma=sig)
        spikes, *_ = model.simulate(Ntrials=TEST_NTRIALS, T=T_BINS)
        pred, shp = classify_spike_trains(spikes, threshold)
        results.append(("ramp", pred, shp, f"β={beta}, σ={sig}"))


hits = sum(true == pred for true, pred, *_ in results)
print(f"Overall accuracy: {hits}/{len(results)} "
      f"({100*hits/len(results):.1f} %)\n")

for true, pred, shp, params in results:
    ok = 'T' if true == pred else 'F'
    print(f"{params:<22s}  sharpness={shp:7.3f}  → predicted {pred.upper()} {ok}")
